# **We use langchain + langgraph**

In [5]:
import re
from typing_extensions import Literal

from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage, ToolMessage
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command, interrupt
from dotenv import load_dotenv

load_dotenv()
llm = init_chat_model("gpt-4o-mini", temperature=0)


# --- 1. Tools (with simple guardrails) ---

@tool
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


@tool
def divide(a: int, b: int) -> float:
    """Divide `a` by `b`."""
    if b == 0:
        # Guardrail: stop division by zero cleanly
        return "Error: cannot divide by zero."
    return a / b


@tool
def calculator(expression: str) -> str:
    """Evaluate a simple math expression like '12 * 4 / 2'."""
    # Guardrail: allow only numbers, spaces, and basic math symbols (+, -, *, /, ., parentheses)
    if not re.fullmatch(r"[0-9+\-*/().\s]+", expression):
        return "Error: expression contains disallowed characters."
    return str(eval(expression, {"__builtins__": {}}, {}))


tools = [add, divide, calculator]
tools_by_name = {t.name: t for t in tools}
llm_with_tools = llm.bind_tools(tools)

# Only ask for human approval on `calculator`
NEEDS_APPROVAL = {"calculator"}


# --- 2. Graph Nodes ---

def llm_call(state: MessagesState) -> dict:
    """Ask the LLM what to do next."""
    system = SystemMessage(content="You are a helpful math assistant. Use tools when needed.")
    reply = llm_with_tools.invoke([system] + state["messages"])
    return {"messages": [reply]}


def tool_node(state: MessagesState) -> dict:
    """Run tools, pausing for human approval when needed."""
    last_message = state["messages"][-1]
    tool_messages = []

    for call in last_message.tool_calls:
        name = call["name"]
        args = call["args"]
        call_id = call["id"]

        # Human-in-the-loop pause
        if name in NEEDS_APPROVAL:
            approved = interrupt(f"Approve running `{name}` with {args}? (yes/no)")
            if str(approved).strip().lower() != "yes":
                tool_messages.append(
                    ToolMessage(content=f"Skipped `{name}` (not approved).", tool_call_id=call_id)
                )
                continue

        # Run the chosen tool
        result = tools_by_name[name].invoke(args)
        tool_messages.append(ToolMessage(content=str(result), tool_call_id=call_id))

    return {"messages": tool_messages}


# --- 3. Routing ---

def should_continue(state: MessagesState) -> Literal["tool_node", END]:
    """Go to `tool_node` if the LLM requested a tool, otherwise finish."""
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tool_node"
    return END


# --- 4. Build & Compile ---

graph_builder = StateGraph(MessagesState)
graph_builder.add_node("llm_call", llm_call)
graph_builder.add_node("tool_node", tool_node)

graph_builder.add_edge(START, "llm_call")
graph_builder.add_conditional_edges("llm_call", should_continue, ["tool_node", END])
graph_builder.add_edge("tool_node", "llm_call")

# A checkpointer is required so `interrupt()` can save and resume state
agent = graph_builder.compile(checkpointer=InMemorySaver())


# --- 5. Run & Resume ---

if __name__ == "__main__":
    config = {"configurable": {"thread_id": "demo-1"}}

    # First run: stops if `interrupt()` is reached
    output = agent.invoke(
        {"messages": [HumanMessage(content="Calculate (12 * 4) / 2")]},
        config=config,
    )

    # Resume loop if paused for human input
    while output.get("__interrupt__"):
        question = output["__interrupt__"][0].value
        answer = input(f"{question}: ")
        output = agent.invoke(Command(resume=answer), config=config)

    for msg in output["messages"]:
        msg.pretty_print()

Approve running `calculator` with {'expression': '(12 * 4) / 2'}? (yes/no):  yes


================================ Human Message =================================

Calculate (12 * 4) / 2
================================== Ai Message ==================================
Tool Calls:
  calculator (call_TsOYCWN0FINJ65z9tq3bxTj8)
 Call ID: call_TsOYCWN0FINJ65z9tq3bxTj8
  Args:
    expression: (12 * 4) / 2
================================= Tool Message =================================

24.0
================================== Ai Message ==================================

The result of the calculation \((12 * 4) / 2\) is \(24\).
